### Árbol de decisión

**Objetivo:** clasificar los registros en dos categorías: sin diabetes (`0`) y con prediabetes o diabetes (`1`).

Este notebook utiliza la división de entrenamiento y prueba preparada por el equipo. Para el árbol se usan las características originales, sin normalizar, porque los árboles de decisión realizan divisiones mediante umbrales y no dependen de distancias entre variables.

Los archivos de entrada son:
- `X_train.csv` y `X_test.csv`: características originales, sin normalizar.
- `y_train.csv` y `y_test.csv`: etiquetas originales de tres clases (`Diabetes_012`).

Como el objetivo de este algoritmo es binario, las etiquetas se transforman así: `0` permanece como 0 y las clases `1` (prediabetes) y `2` (diabetes) se convierten en 1.

Se utiliza `class_weight="balanced"` para compensar el desbalance de las dos clases resultantes. Los hiperparámetros se seleccionan con validación cruzada sobre el conjunto de entrenamiento, sin volver a dividir los datos del equipo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, json
from pathlib import Path

# Parámetros del proyecto. Si params.yaml no está disponible, se usa una semilla fija.
try:
    with open("../params.yaml") as f:
        params = yaml.safe_load(f)
    SEED = params.get("semilla", 42)
except FileNotFoundError:
    SEED = 42

FIG = Path("../reports/figures/03_arbol_decision")
FIG.mkdir(parents=True, exist_ok=True)
Path("../metrics").mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve, accuracy_score, precision_score,
                             recall_score, f1_score)

In [ ]:
# Cargar la partición compartida por el equipo
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train_original = pd.read_csv("../data/processed/y_train.csv")["Diabetes_012"]
y_test_original = pd.read_csv("../data/processed/y_test.csv")["Diabetes_012"]

# Verificar que las filas de características y etiquetas estén alineadas
assert len(X_train) == len(y_train_original), "X_train y y_train tienen distinto número de filas"
assert len(X_test) == len(y_test_original), "X_test y y_test tienen distinto número de filas"
assert X_train.columns.tolist() == X_test.columns.tolist(), "Las columnas de train y test no coinciden"

# Convertir la variable objetivo de tres clases a clasificación binaria:
# 0 = sin diabetes; 1 = prediabetes o diabetes
y_train = (y_train_original > 0).astype(int)
y_test = (y_test_original > 0).astype(int)

# Confirmar que los patrones de características no se repitan entre train y test
# (esto verifica X; no identifica personas, porque no hay ID individual en estos archivos).
train_patterns = set(map(tuple, X_train.to_numpy()))
test_patterns = set(map(tuple, X_test.to_numpy()))
patrones_compartidos = len(train_patterns.intersection(test_patterns))

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Distribución binaria train (%):")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))
print("Distribución binaria test (%):")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))
print("Patrones de características compartidos entre train y test:", patrones_compartidos)

assert patrones_compartidos == 0, (
    "Hay patrones de características compartidos entre train y test. "
    "Revisar cómo se generó la partición del equipo."
)

#### 1. Búsqueda de hiperparámetros (validación cruzada agrupada de 3 particiones)

Se utiliza `StratifiedGroupKFold` dentro del conjunto de entrenamiento. Los registros con el mismo patrón de características se mantienen dentro de una misma partición de validación. La división externa de entrenamiento y prueba se conserva tal como la preparó el equipo.

In [ ]:
# Crear grupos a partir de patrones idénticos de características en entrenamiento
groups_train = pd.Series(
    pd.MultiIndex.from_frame(X_train).factorize(sort=False)[0],
    index=X_train.index
)

cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

grid = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=SEED),
    param_grid={
        "max_depth": [3, 5, 7, 9, 12],
        "min_samples_leaf": [50, 200, 500]
    },
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)

# La validación cruzada utiliza únicamente X_train e y_train.
grid.fit(X_train, y_train, groups=groups_train)

print("Mejores parámetros:", grid.best_params_)
print("AUC promedio de validación cruzada:", round(grid.best_score_, 4))

res = pd.DataFrame(grid.cv_results_)[[
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_train_score",
    "mean_test_score"
]].round(4)

res.sort_values("mean_test_score", ascending=False).head(8)

In [ ]:
pivot = res.pivot(
    index="param_max_depth",
    columns="param_min_samples_leaf",
    values="mean_test_score"
).astype(float)

plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.title("AUC de validación cruzada")
plt.xlabel("Mínimo de muestras por hoja")
plt.ylabel("Profundidad máxima")
plt.savefig(FIG / "grid_busqueda.png", dpi=120, bbox_inches="tight")
plt.show()

#### 2. Evaluación en el conjunto de prueba

In [ ]:
# Evaluar el mejor modelo en el conjunto de prueba reservado por el equipo
arbol = grid.best_estimator_
y_pred = arbol.predict(X_test)
y_prob = arbol.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test,
    y_pred,
    target_names=["Sin diabetes", "Prediabetes/Diabetes"],
    zero_division=0
))

auc = roc_auc_score(y_test, y_prob)
print("AUC en prueba:", round(auc, 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=["No", "Sí"]
).plot(ax=axes[0], cmap="Blues", values_format="d")
axes[0].set_title("Matriz de confusión")

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
axes[1].set(
    xlabel="Tasa de falsos positivos",
    ylabel="Tasa de verdaderos positivos",
    title="Curva ROC"
)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG / "evaluacion.png", dpi=120, bbox_inches="tight")
plt.show()

#### 2.1 Comparación con un modelo base

Para contextualizar el desempeño, se compara el árbol con un clasificador muy sencillo que siempre predice la clase mayoritaria del conjunto de entrenamiento. En este dataset, esa clase es `0` (sin diabetes).

Este modelo base sirve como referencia: puede obtener una accuracy alta por el desbalance de clases, pero no identifica casos positivos. La comparación ayuda a explicar por qué no se debe evaluar el árbol únicamente con accuracy.

In [ ]:
# Modelo base: siempre predice la clase mayoritaria del entrenamiento
clase_mayoritaria = int(y_train.mode().iloc[0])
y_pred_base = np.full(shape=len(y_test), fill_value=clase_mayoritaria)

accuracy_base = accuracy_score(y_test, y_pred_base)
recall_base = recall_score(y_test, y_pred_base, zero_division=0)

accuracy_arbol = accuracy_score(y_test, y_pred)
recall_arbol = recall_score(y_test, y_pred, zero_division=0)

comparacion_base = pd.DataFrame({
    "Modelo": ["Predicción de clase mayoritaria", "Árbol de decisión"],
    "Accuracy": [accuracy_base, accuracy_arbol],
    "Recall clase positiva": [recall_base, recall_arbol]
})

comparacion_base.round(4)

##### Interpretación de resultados

En la prueba se obtuvo un **AUC de 0.8171**, mientras que el AUC promedio de validación cruzada fue **0.8121**. La diferencia es pequeña (0.005), por lo que el desempeño en prueba es consistente con el observado durante la selección de hiperparámetros.

Las métricas del conjunto de prueba fueron:

| Métrica | Resultado | Interpretación |
|---|---:|---|
| Accuracy | 0.7080 | El 70.8% de las predicciones fueron correctas. |
| Precision | 0.3242 | De las predicciones positivas, el 32.42% fue correcto. |
| Recall | 0.7868 | Se identificó el 78.68% de los casos positivos reales. |
| F1-score | 0.4592 | Resume el equilibrio entre precision y recall. |
| AUC | 0.8171 | El modelo tiene buena capacidad para distinguir las dos clases. |

La matriz de confusión muestra:
- **29,632 verdaderos negativos:** personas sin diabetes clasificadas correctamente.
- **13,112 falsos positivos:** personas sin diabetes clasificadas como positivas.
- **1,705 falsos negativos:** personas positivas clasificadas como negativas.
- **6,291 verdaderos positivos:** personas con prediabetes o diabetes clasificadas correctamente.

La comparación con el modelo base muestra por qué la accuracy debe analizarse con cuidado. El modelo base que siempre predice la clase mayoritaria puede alcanzar aproximadamente **84.24% de accuracy**, pero su recall para la clase positiva es **0%**: no detecta ningún registro positivo. El árbol obtiene una accuracy menor (**70.80%**), pero identifica el **78.68%** de los casos positivos. Por tanto, no es correcto concluir que el modelo base sea mejor usando solo accuracy; ambos tienen comportamientos muy distintos y la elección de métricas depende del objetivo del proyecto.

El uso de `class_weight="balanced"` da mayor peso a la clase minoritaria y puede favorecer la detección de positivos, aunque también puede aumentar los falsos positivos. En este modelo se observa precisamente ese intercambio: recall relativamente alto y precision baja.

Estos resultados describen el rendimiento en este conjunto de prueba; no significan que el modelo pueda diagnosticar diabetes en personas reales.

#### 3. Selección de variables e importancia

Para este algoritmo se utilizan las **21 variables predictoras** disponibles en `X_train.csv` y `X_test.csv`.

La decisión de conservarlas se basa en tres razones:

1. El equipo preparó una partición común para que los algoritmos trabajen con los mismos registros y puedan compararse de forma consistente.
2. El análisis exploratorio del notebook de limpieza (información mutua y correlación de Spearman) permite identificar asociaciones iniciales, pero no se estableció un criterio validado para eliminar variables.
3. El árbol de decisión realiza una selección interna durante el entrenamiento: elige las variables y los puntos de corte que mejor separan las clases en cada nodo. Por eso, no es necesario eliminar de antemano todas las variables con asociación individual baja.

La gráfica de importancia que se presenta a continuación es una **interpretación posterior del modelo**, no una prueba de causalidad ni una selección automática definitiva de variables.

**Normalización:** se utilizan las características originales, sin Z-score. Los árboles de decisión dividen los datos mediante umbrales y no dependen de distancias ni de que las variables tengan la misma escala.

In [ ]:
imp = pd.Series(arbol.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(7, 7))
imp.plot.barh(color="steelblue")
plt.title("Importancia de variables (árbol de decisión)")
plt.xlabel("Importancia")
plt.tight_layout()
plt.savefig(FIG / "importancia.png", dpi=120, bbox_inches="tight")
plt.show()

imp.sort_values(ascending=False).head(8).round(4)

#### 4. Visualización de un árbol interpretable (profundidad 3)

El árbol optimizado puede ser demasiado complejo para interpretarlo visualmente. Por eso se entrena un segundo árbol limitado a profundidad 3. Esta versión permite observar reglas generales, pero no sustituye al modelo optimizado utilizado para la evaluación principal.

In [ ]:
arbol3 = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=500,
    class_weight="balanced",
    random_state=SEED
).fit(X_train, y_train)

plt.figure(figsize=(20, 9))
plot_tree(
    arbol3,
    feature_names=X_train.columns,
    class_names=["No", "Sí"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.savefig(FIG / "arbol_profundidad3.png", dpi=120, bbox_inches="tight")
plt.show()

auc_arbol3 = roc_auc_score(y_test, arbol3.predict_proba(X_test)[:, 1])
print("AUC árbol profundidad 3:", round(auc_arbol3, 4))

In [ ]:
metricas = {
    "mejores_parametros": {
        k: (int(v) if isinstance(v, (np.integer,)) else v)
        for k, v in grid.best_params_.items()
    },
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "accuracy_modelo_base": round(float(accuracy_base), 4),
    "recall_modelo_base": round(float(recall_base), 4),
    "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
    "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
    "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
    "auc": round(auc, 4),
    "auc_validacion_cruzada": round(float(grid.best_score_), 4),
    "auc_arbol_profundidad_3": round(float(auc_arbol3), 4),
    "datos_entrada": [
        "X_train.csv",
        "X_test.csv",
        "y_train.csv",
        "y_test.csv"
    ],
    "normalizacion": "No aplicada; árbol de decisión utiliza características originales"
}

with open("../metrics/arbol_decision.json", "w") as f:
    json.dump(metricas, f, indent=2)

metricas

##### Interpretación de resultados

Las cinco variables con mayor importancia en esta ejecución fueron:

| Variable | Importancia |
|---|---:|
| `GenHlth` | 0.4722 (47.22%) |
| `HighBP` | 0.2365 (23.65%) |
| `BMI` | 0.1113 (11.13%) |
| `Age` | 0.1027 (10.27%) |
| `HighChol` | 0.0457 (4.57%) |

En conjunto representan aproximadamente el **96.84%** de la importancia del árbol. `GenHlth` (estado general de salud) es la variable con mayor importancia en este modelo, seguida de `HighBP` (presión arterial alta), `BMI` (índice de masa corporal), `Age` (grupo de edad) y `HighChol` (colesterol alto).

La importancia indica cuánto contribuyó cada variable a las divisiones del árbol. No significa que esas variables causen diabetes ni que las demás carezcan por completo de utilidad.

El árbol de profundidad 3 obtuvo un **AUC de 0.7759**, frente a **0.8171** del árbol optimizado. La versión sencilla facilita explicar las reglas principales, pero pierde parte de la capacidad de discriminación. En la visualización, la primera división se realiza con `GenHlth <= 2.5`; después aparecen variables como `HighBP`, `Age` y `BMI`.

**Limitación:** la variable objetivo agrupa prediabetes y diabetes en una sola clase positiva. Por ello, el modelo no distingue entre prediabetes y diabetes; clasifica entre ausencia de diabetes y presencia de prediabetes o diabetes. Es un ejercicio académico y no una herramienta de diagnóstico médico.

#### 5. Consideraciones éticas y limitaciones

Este modelo utiliza información relacionada con la salud y, por ello, sus resultados deben manejarse con precaución.

- **Privacidad:** los archivos deben utilizarse únicamente dentro del proyecto académico y no deben publicarse datos personales identificables. No se debe intentar identificar a las personas representadas en los registros.
- **Sesgos y representatividad:** los datos provienen de una encuesta y pueden no representar por igual a todas las poblaciones. El desempeño observado no garantiza los mismos resultados para otros grupos o contextos.
- **Errores de clasificación:** los falsos positivos pueden generar preocupación innecesaria y los falsos negativos pueden dar una falsa sensación de seguridad. Por eso, las predicciones no deben utilizarse para tomar decisiones médicas.
- **Alcance de la etiqueta:** el objetivo agrupa prediabetes y diabetes en una sola clase positiva; el modelo no distingue entre ambas condiciones.
- **Asociación no significa causalidad:** la importancia de una variable indica su contribución a las divisiones del árbol, pero no demuestra que cause diabetes.

El árbol se presenta exclusivamente como un ejercicio académico de análisis y clasificación de datos. No sustituye una evaluación clínica ni debe utilizarse como herramienta de diagnóstico.

#### 6. Conclusión del árbol de decisión

Se entrenó un árbol de decisión para clasificar los registros en dos categorías: sin diabetes y con prediabetes o diabetes. Se utilizaron las características originales sin normalizar, porque este algoritmo no requiere estandarización. También se aprovechó la partición de entrenamiento y prueba preparada por el equipo, sin realizar una nueva separación.

La búsqueda de hiperparámetros seleccionó `max_depth=9` y `min_samples_leaf=200`, con un AUC promedio de validación cruzada de **0.8121**. En el conjunto de prueba se obtuvo un AUC de **0.8171**, un recall de **0.7868**, una precision de **0.3242** y un F1-score de **0.4592**. Esto muestra que el modelo logra identificar una proporción importante de los casos positivos, aunque genera una cantidad considerable de falsos positivos.

Las variables con mayor importancia fueron `GenHlth`, `HighBP`, `BMI`, `Age` y `HighChol`. El árbol de profundidad 3 obtuvo un AUC de **0.7759**: es más sencillo de interpretar, pero tiene menor desempeño que el modelo optimizado.

En conclusión, el árbol de decisión es pertinente para este problema porque permite clasificar los registros y explicar las reglas utilizadas. Su principal limitación observada es la precision relativamente baja, por lo que sus predicciones positivas deben interpretarse con cautela. Además, al combinar prediabetes y diabetes en una sola clase, el modelo no diferencia entre ambas condiciones y no debe utilizarse como diagnóstico médico.